# Phase 1 — EDA & Data Audit: Niên giám Thống kê 2025 & Biểu tháng 10/2026

**Người thực hiện:** Thành viên 1 (Long — Data & Benchmark Lead)  
**Mục tiêu:** Khám phá cấu trúc thật của 2 tệp nguồn chính thức (`NG-TOAN-QUOC-2025-1.pdf` và `2.-Bieu-thang-10.2026.xlsx`) và xuất báo cáo kiểm toán tái lập `docs/data_audit.json`:
1. Kiểm toán Mục lục (TOC) PDF 1.557 trang (`cover_offset=1`) và 20 sheet Excel.
2. Quét tự động danh mục biểu của 4 chương mục tiêu (173 biểu tiếng Việt) → đối chiếu với 32 bảng chọn lọc trong `docs/table_selection.yaml`.
3. Phát hiện cấu trúc hành chính (63 tỉnh trong Niên giám 2025 vs. 34 tỉnh sau sáp nhập trong Biểu 10/2026).
4. Xuất `docs/data_audit.json` làm đầu vào cho Task 2 (`admin_mapping`) và Task 3 (`pdf_parser`/`excel_parser`).

In [1]:
from pathlib import Path
from veristat.data.audit import extract_pdf_toc, audit_excel_workbook, generate_data_audit

ROOT = Path('..') if Path('../NG-TOAN-QUOC-2025-1.pdf').exists() else Path('.')
PDF_PATH = ROOT / 'NG-TOAN-QUOC-2025-1.pdf'
XLSX_PATH = ROOT / '2.-Bieu-thang-10.2026.xlsx'
SELECTION_YAML = ROOT / 'docs/table_selection.yaml'
AUDIT_JSON = ROOT / 'docs/data_audit.json'

toc = extract_pdf_toc(PDF_PATH, cover_offset=1)
sheets = audit_excel_workbook(XLSX_PATH)
print(f'PDF TOC entries: {len(toc)} | Excel sheets: {len(sheets)}')
for item in toc:
    print(f"  [L{item['level']}] book p.{item['page']} (pdf p.{item['pdf_page']}): {item['title']}")

PDF TOC entries: 14 | Excel sheets: 20
  [L1] book p.1 (pdf p.2): 00. LND Tong quan chung 2025_b3
  [L1] book p.41 (pdf p.42): 01. DV hanh chinh- Dat dai-Khi hau 2025_b3
  [L1] book p.61 (pdf p.62): 02. Dan so - Lao dong (2025)_b3
  [L1] book p.193 (pdf p.194): 03. Tai khoan Quoc gia (2025)_b3
  [L1] book p.261 (pdf p.262): 04. CN, Dau tu va xay dung (2025)_b3
  [L1] book p.345 (pdf p.346): 05. Doanh nghiep (2025)_b3
  [L1] book p.525 (pdf p.526): 06. Nong nghiep (2025)_b3
  [L1] book p.663 (pdf p.664): 07. Thuong mai, du lich (2025)_b3
  [L1] book p.723 (pdf p.724): 08. Chi so gia (2025)_b3
  [L1] book p.785 (pdf p.786): 09. Van tai - BCVT (2025)_b3
  [L1] book p.829 (pdf p.830): 10. Giao duc (2025)_b3
  [L1] book p.857 (pdf p.858): 11. Y te, VH-TT, Muc song dan cu (2025)_b3
  [L1] book p.989 (pdf p.990): 12. Thong ke nuoc ngoai (2025)_b3
  [L1] book p.1141 (pdf p.1142): 13. Phu luc (2025)_b3


In [2]:
# Chạy toàn bộ Data Audit và xuất trực tiếp docs/data_audit.json
audit = generate_data_audit(
    pdf_path=PDF_PATH,
    xlsx_path=XLSX_PATH,
    selection_yaml=SELECTION_YAML,
    out_json=AUDIT_JSON,
    cover_offset=1,
)
print('Auto-scanned PDF tables:', audit['auto_scanned_pdf_catalog']['total_tables_scanned'])
print('By chapter:', audit['auto_scanned_pdf_catalog']['by_chapter'])
print('Curated summary:', audit['curated_summary'])

Auto-scanned PDF tables: 173
By chapter: {'dan_so_lao_dong': {'total': 60, 'with_geo': 34}, 'doanh_nghiep': {'total': 46, 'with_geo': 13}, 'giao_duc': {'total': 11, 'with_geo': 0}, 'y_te_msdc': {'total': 56, 'with_geo': 38}}
Curated summary: {'total_tables': 32, 'by_domain': {'dan_so': 6, 'lao_dong': 6, 'giao_duc': 6, 'y_te': 6, 'doanh_nghiep': 8}, 'by_source': {'pdf_2025': 27, 'excel_2026': 5}}


In [3]:
# Kiểm tra danh sách 34 tỉnh/thành mới trong Excel 10/2026 (sau sáp nhập 2025)
excel_geo = audit['findings']['excel_2026_geo_entities']
print(f'Excel 5. LĐCN_DP geo entries ({len(excel_geo)}):')
print(excel_geo)

Excel 5. LĐCN_DP geo entries (35):
['CẢ NƯỚC', 'Hà Nội', 'Bắc Ninh', 'Quảng Ninh', 'Hải Phòng', 'Hưng Yên', 'Ninh Bình', 'Cao Bằng', 'Tuyên Quang', 'Lào Cai', 'Thái Nguyên', 'Lạng Sơn', 'Phú Thọ', 'Điện Biên', 'Lai Châu', 'Sơn La', 'Thanh Hóa', 'Nghệ An', 'Hà Tĩnh', 'Quảng Trị', 'Huế', 'Đà Nẵng', 'Quảng Ngãi', 'Gia Lai', 'Khánh Hòa', 'Đắk Lắk', 'Lâm Đồng', 'Tây Ninh', 'Đồng Nai', 'TP. Hồ Chí Minh', 'Vĩnh Long', 'Đồng Tháp', 'An Giang', 'Cần Thơ', 'Cà Mau']
